# 02 — Exploração da base TOTVS para RAG

A base já está entregue como JSON em `data/knowledge_base/totvs_rag_kb_v1.json`. Este notebook apenas carrega, valida e explora o artefato pronto.

**Objetivo e conexão com o projeto.** Esta etapa examina a base TOTVS que dará contexto aos resultados do classificador e verifica se cada documento possui o contrato mínimo esperado pelo retrieval. A busca lexical apresentada aqui funciona como referência inicial: ela permite validar conteúdo, metadados e fontes antes de introduzir métodos semânticos mais complexos.


## 1. Decisões tomadas

- O JSON contém uma lista de 107 chunks independentes e não depende de um conversor no projeto.
- Cada chunk mantém ID, tipo, produto, categoria, listas de metadados, nível de evidência, conteúdo, fontes e data de compilação.
- O nível de evidência será usado futuramente para impedir que hipóteses comerciais sejam apresentadas como fatos.
- A busca lexical abaixo é apenas um baseline transparente; embeddings e reranking serão comparados em uma próxima etapa.
- Resultados mostram metadados e fontes, evitando despejar todo o conteúdo na saída do notebook.

As escolhas abaixo delimitam o experimento e devem ser lidas antes da implementação. Elas registram o que o código efetivamente faz e quais limitações precisam permanecer visíveis na interpretação.


## 2. Preparação do ambiente

Nesta célula são reunidos imports, parâmetros e caminhos necessários à etapa. Centralizar essa preparação antes do processamento deixa as dependências explícitas e evita que configurações importantes fiquem dispersas entre transformações, treinamento ou avaliação.


In [5]:
from collections import Counter
from pathlib import Path
import json, re, unicodedata

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
KB_PATH = PROJECT_ROOT / 'data' / 'knowledge_base' / 'totvs_rag_kb_v1.json'
records = json.loads(KB_PATH.read_text(encoding='utf-8'))
len(records)

107

## 3. Validação e auditoria

Validamos o contrato mínimo, unicidade dos IDs e presença de fontes antes de usar a base em recuperação.

As verificações desta seção funcionam como critérios de integridade. Elas não melhoram as métricas por si mesmas, mas impedem que a análise prossiga silenciosamente com IDs repetidos, campos ausentes ou relações inconsistentes.


In [6]:
required_fields = {'id', 'title', 'document_type', 'company', 'product', 'category', 'evidence_level', 'content', 'sources', 'compiled_at'}
assert isinstance(records, list) and records
assert all(required_fields <= record.keys() for record in records)
assert len({record['id'] for record in records}) == len(records)
assert all(record['sources'] for record in records)
{
    'total_chunks': len(records),
    'unique_ids': len({record['id'] for record in records}),
    'document_types': dict(Counter(record['document_type'] for record in records)),
    'evidence_levels': dict(Counter(record['evidence_level'] for record in records)),
    'unique_source_urls': len({source['url'] for record in records for source in record['sources']}),
}

{'total_chunks': 107,
 'unique_ids': 107,
 'document_types': {'visao_geral': 1,
  'produto': 31,
  'comparacao_erp': 4,
  'comparacao_concorrente': 10,
  'glossario': 30,
  'dor_para_produto': 18,
  'sinal_oportunidade': 10,
  'taxonomia': 1,
  'dicionario_concorrentes': 1,
  'arquitetura_rag': 1},
 'evidence_levels': {'FATO DOCUMENTADO': 30,
  'VALIDAÇÃO RECOMENDADA': 1,
  'FATO DOCUMENTADO / VALIDAÇÃO COMERCIAL': 1,
  'FATO + INFERÊNCIA EXPLÍCITA': 14,
  'FATO CONCEITUAL + CONTEXTO COMERCIAL': 30,
  'HIPÓTESE COMERCIAL EXPLÍCITA': 28,
  'RECOMENDAÇÃO DE MODELAGEM': 1,
  'FATO NOMINAL + RECOMENDAÇÃO DE NER': 1,
  'RECOMENDAÇÃO DE ARQUITETURA': 1},
 'unique_source_urls': 52}

## 4. Baseline de busca lexical

Esta função mede sobreposição de termos em título, conteúdo e palavras-chave. Ela servirá como referência simples para avaliar se embeddings realmente melhoram a recuperação.


In [ ]:
def normalize_for_search(text):
    text = unicodedata.normalize('NFKD', text.casefold())
    return ''.join(char for char in text if not unicodedata.combining(char))

def lexical_search(query, top_k=5):
    query_terms = set(re.findall(r'\w+', normalize_for_search(query)))
    ranked = []
    for record in records:
        searchable = ' '.join((record['title'], record['content'], ' '.join(record['keywords'])))
        terms = set(re.findall(r'\w+', normalize_for_search(searchable)))
        score = len(query_terms & terms)
        if score:
            ranked.append((score, record))
    ranked.sort(key=lambda item: (-item[0], item[1]['id']))
    return [
        {'score': score, 'id': record['id'], 'title': record['title'], 'evidence_level': record['evidence_level'], 'source_urls': [source['url'] for source in record['sources']]}
        for score, record in ranked[:top_k]
    ]


### Execução da etapa

A célula aplica os objetos e funções preparados anteriormente a uma responsabilidade específica do fluxo. As saídas permanecem disponíveis para validação e interpretação nas células seguintes.


In [7]:
lexical_search('estoque divergente entre filiais e separação no armazém')


[{'score': 5,
  'id': 'TOTVS-PROD-012',
  'title': 'TOTVS WMS – Linha Logix',
  'evidence_level': 'FATO DOCUMENTADO',
  'source_urls': ['https://produtos.totvs.com/ficha-tecnica/tudo-sobre-o-totvs-wms-linha-logix/',
   'https://www.totvs.com/wms/']},
 {'score': 4,
  'id': 'TOTVS-PAIN-001',
  'title': 'Dor → produto: estoque_filiais',
  'evidence_level': 'HIPÓTESE COMERCIAL EXPLÍCITA',
  'source_urls': ['https://ri.totvs.com/a-companhia/segmentos-e-produtos/']},
 {'score': 4,
  'id': 'TOTVS-PAIN-010',
  'title': 'Dor → produto: armazem',
  'evidence_level': 'HIPÓTESE COMERCIAL EXPLÍCITA',
  'source_urls': ['https://ri.totvs.com/a-companhia/segmentos-e-produtos/']},
 {'score': 4,
  'id': 'TOTVS-PROD-005',
  'title': 'TOTVS Distribuição e Varejo – Linha WinThor',
  'evidence_level': 'FATO DOCUMENTADO',
  'source_urls': ['https://www.totvs.com/distribuicao/totvs-distribuicao-e-varejo/',
   'https://produtos.totvs.com/ficha-tecnica/tudo-sobre-o-totvs-distribuicao-e-varejo-linha-winthor/']},